# Sp500 index daily

link: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks?select=sp500_index.csv

In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import plotly.graph_objects as go

In [2]:
def TimeSeriesChart(real_data, predictions, title="Backtesting (Actual vs Forecast)"):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=real_data.index,
            y=real_data[TARGET],
            name="Actual S&P500 Price",
            mode="lines",
            line=dict(color="#1f77b4", width=2),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["mean"],
            name="Forecast (Mean)",
            mode="lines",
            line=dict(color="#ff7f0e", width=2, dash="dash"),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.5"],
            name="Forecast (Median)",
            mode="lines",
            line=dict(color="#000000", width=2, dash="dash"),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.975"],
            name="Upper Bound (p97.5)",
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.025"],
            name="95% Confidence Interval (p2.5 - p97.5)",
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(255, 127, 14, 0.15)",
            line=dict(width=0),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.95"],
            name="Upper Bound (p95)",
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.05"],
            name="90% Confidence Interval (p5 - p95)",
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(200, 100, 14, 0.15)",
            line=dict(width=0),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.9"],
            name="Upper Bound (p90)",
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=predictions.index,
            y=predictions["0.1"],
            name="80% Confidence Interval (p10 - p90)",
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(100, 60, 14, 0.15)",
            line=dict(width=0),
        )
    )

    fig.update_layout(
        title=f"{title}",
        xaxis_title="Date",
        yaxis_title="S&P 500 Index Price ($)",
        template="plotly_white",
        hovermode="x unified",
    )
    fig.show()

In [3]:
df = pd.read_csv("../data/01_raw/sp500_index.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df = df.set_index("Date").asfreq("D").ffill().reset_index()
df["item_id"] = "SP500"
df

,Date,S&P500,item_id
0,2014-12-22,2078.54,SP500
1,2014-12-23,2082.17,SP500
2,2014-12-24,2081.88,SP500
3,2014-12-25,2081.88,SP500
4,2014-12-26,2088.77,SP500
...,...,...,...
3647,2024-12-16,6074.08,SP500
3648,2024-12-17,6050.61,SP500
3649,2024-12-18,5872.16,SP500
3650,2024-12-19,5867.08,SP500


In [4]:
QUANTILE_LEVELS = [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]
TARGET = "S&P500"
EVAL_METRIC = "MAE"
MODELS = {
    "RecursiveTabular": {"tabular_hyperparameters": {"GBM": {}, "XGB": {}, "CAT": {}}},
    "Naive": {},
    "SeasonalNaive": {},
    "PatchTST": {},
    "ETS": {},
    "ADIDA": {},
    "DLinear": {},
    "Toto": {},
    "ARIMA": {},
    "Theta": {},
    "DeepAR": {},
    "NPTS": {},
    "TemporalFusionTransformer": {},
    "Chronos2": {},
    "SimpleFeedForward": {},
    "TiDE": {},
    "WaveNet": {},
}

In [5]:
PREDICTION_LENGTH = 30

In [6]:
train_data = TimeSeriesDataFrame.from_data_frame(
    df, id_column="item_id", timestamp_column="Date"
)

train_split, test_split = train_data.train_test_split(
    prediction_length=PREDICTION_LENGTH
)

predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=QUANTILE_LEVELS,
    target=TARGET,
    eval_metric=EVAL_METRIC,
)

predictor.fit(
    train_split,
    hyperparameters=MODELS
)

Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/ai_model_evaluation_mlops/notebooks/AutogluonModels/ag-20260518_184325'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.62/11.62 GB
Total GPU Memory:   Free: 11.62 GB, Allocated: 0.00 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       25.30 GB / 31.15 GB (81.2%)
Disk Space Avail:   111.64 GB / 230.30 GB (48.5%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                     'DLinear': {},
                     'DeepAR': {},
                     'ETS':

In [7]:
leaderboard = predictor.leaderboard(train_split)

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


In [8]:
leaderboard

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-62.082486,-61.754528,0.108112,0.086437,0.225190,18
1,PatchTST,-65.531157,-65.531157,0.015862,0.006576,19.452815,10
2,SimpleFeedForward,-66.431859,-66.431859,0.009032,0.005759,11.292627,16
3,DLinear,-66.864831,-66.864831,0.009851,0.006122,20.475153,14
4,Chronos2,-71.126419,-71.126419,0.901200,0.260729,2.410235,7
5,TiDE,-71.715156,-71.715156,0.018974,0.007679,55.017353,11
6,TemporalFusionTransformer,-73.194122,-73.194122,0.035776,0.008278,19.429251,8
7,RecursiveTabular,-77.578291,-77.578291,0.065837,0.060269,0.386214,3
8,DeepAR,-78.894491,-75.611035,0.072364,0.066720,41.733151,9
9,Naive,-80.265667,-80.265667,0.781923,0.924211,0.006185,1


In [9]:
best_model = leaderboard.loc[0]
best_model = best_model["model"]
best_model

'WeightedEnsemble'

In [10]:
predictions = predictor.predict(train_split, model=best_model)

In [11]:
selected_item = "SP500"

real_data = test_split.loc[selected_item]
predictions_store = predictions.loc[selected_item]

In [12]:
TimeSeriesChart(real_data,predictions_store, "Daily Backtesting - SP500")

## Weekly Analysis & Forecasting

In this section, we group the daily data by week, using the value from the start of the week (Monday) and omitting the other days, in order to perform a weekly forecast analysis.

In [13]:
df_weekly = df.set_index("Date").resample("W-MON").first().reset_index()

df_weekly["item_id"] = "SP500_weekly"

df_weekly.head(10)

,Date,S&P500,item_id
0,2014-12-22,2078.54,SP500_weekly
1,2014-12-29,2082.17,SP500_weekly
2,2015-01-05,2080.35,SP500_weekly
3,2015-01-12,2002.61,SP500_weekly
4,2015-01-19,2023.03,SP500_weekly
5,2015-01-26,2022.55,SP500_weekly
6,2015-02-02,2029.55,SP500_weekly
7,2015-02-09,2050.03,SP500_weekly
8,2015-02-16,2068.59,SP500_weekly
9,2015-02-23,2100.34,SP500_weekly


In [14]:
PREDICTION_LENGTH = 13

In [15]:
train_data_weekly = TimeSeriesDataFrame.from_data_frame(
    df_weekly, id_column="item_id", timestamp_column="Date"
)

train_split_weekly, test_split_weekly = train_data_weekly.train_test_split(
    prediction_length=PREDICTION_LENGTH
)


predictor_weekly = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=QUANTILE_LEVELS,
    target=TARGET,
    eval_metric=EVAL_METRIC,
)

predictor_weekly.fit(
    train_split_weekly,
    hyperparameters=MODELS
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260518_184701"
Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/ai_model_evaluation_mlops/notebooks/AutogluonModels/ag-20260518_184701'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 10.59/11.62 GB
Total GPU Memory:   Free: 10.59 GB, Allocated: 1.03 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       22.10 GB / 31.15 GB (70.9%)
Disk Space Avail:   111.64 GB / 230.30 GB (48.5%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                 

In [16]:
leaderboard_weekly = predictor_weekly.leaderboard(train_split_weekly)
leaderboard_weekly

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-65.248869,-62.774549,0.059592,0.041728,0.220744,18
1,DLinear,-72.643676,-72.643676,0.008806,0.004966,9.322561,14
2,SimpleFeedForward,-78.010197,-78.010197,0.007833,0.005264,15.539845,16
3,ETS,-82.209462,-82.209462,0.013453,0.012945,0.004703,4
4,TiDE,-86.186866,-86.186866,0.016571,0.006616,31.987476,11
5,PatchTST,-87.746307,-87.746307,0.013930,0.005712,10.249526,10
6,ARIMA,-88.936712,-88.936712,0.013293,0.295664,0.004821,15
7,RecursiveTabular,-89.278562,-89.278562,0.045833,0.035861,0.273670,3
8,Theta,-89.288263,-89.288263,0.295709,0.013124,0.004740,6
9,SeasonalNaive,-90.425385,-90.425385,0.013097,0.012772,0.004708,2


In [17]:
best_model = leaderboard_weekly.loc[0]
best_model = best_model["model"]
best_model

'WeightedEnsemble'

In [18]:
predictions_weekly = predictor_weekly.predict(
    train_split_weekly, model=best_model
)

In [19]:
selected_item_weekly = "SP500_weekly"

real_data_weekly = test_split_weekly.loc[selected_item_weekly]
predictions_weekly_store = predictions_weekly.loc[selected_item_weekly]

In [20]:
TimeSeriesChart(real_data_weekly, predictions_weekly_store, "Weekly Backtesting - SP500")

## Monthly Analysis & Forecasting

In this section, we resample the daily S&P 500 index data to a monthly frequency. We select the first available trading day's value of each month and skip the remaining days. We then train an AutoGluon TimeSeriesPredictor to forecast the index price at a monthly resolution.

In [21]:
df_monthly = df.set_index("Date").resample("MS").first().reset_index()

df_monthly["item_id"] = "SP500_monthly"

df_monthly.head(12)

,Date,S&P500,item_id
0,2014-12-01,2078.54,SP500_monthly
1,2015-01-01,2058.90,SP500_monthly
2,2015-02-01,1994.99,SP500_monthly
3,2015-03-01,2104.50,SP500_monthly
4,2015-04-01,2059.69,SP500_monthly
5,2015-05-01,2108.29,SP500_monthly
6,2015-06-01,2111.73,SP500_monthly
7,2015-07-01,2077.42,SP500_monthly
8,2015-08-01,2103.84,SP500_monthly
9,2015-09-01,1913.85,SP500_monthly


In [22]:
PREDICTION_LENGTH = 12

In [23]:
train_data_monthly = TimeSeriesDataFrame.from_data_frame(
    df_monthly, id_column="item_id", timestamp_column="Date"
)

train_split_monthly, test_split_monthly = train_data_monthly.train_test_split(
    prediction_length=PREDICTION_LENGTH
)

predictor_monthly = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=QUANTILE_LEVELS,
    target=TARGET,
    eval_metric=EVAL_METRIC,
)

predictor_monthly.fit(
    train_split_monthly,
    hyperparameters=MODELS
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260518_184918"
Beginning AutoGluon training...
AutoGluon will save models to '/home/cezartdev/Documents/cezartdev/personal/ai_model_evaluation_mlops/notebooks/AutogluonModels/ag-20260518_184918'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Mar  5 00:10:35 UTC 2026
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 11.61/11.62 GB
Total GPU Memory:   Free: 11.61 GB, Allocated: 0.01 GB, Total: 11.62 GB
GPU Count:          1
Memory Avail:       22.14 GB / 31.15 GB (71.1%)
Disk Space Avail:   111.64 GB / 230.30 GB (48.5%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'hyperparameters': {'ADIDA': {},
                     'ARIMA': {},
                     'Chronos2': {},
                 

In [24]:
leaderboard_monthly = predictor_monthly.leaderboard(train_split_monthly)
leaderboard_monthly

Additional data provided, testing on additional data. Resulting leaderboard will be sorted according to test score (`score_test`).


,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-137.119312,-137.119312,0.927718,0.411823,0.222001,18
1,Theta,-202.232874,-202.232874,0.013976,0.013044,0.004941,6
2,Naive,-239.781667,-239.781667,0.014282,0.014335,0.005309,1
3,TemporalFusionTransformer,-247.519365,-247.519365,0.030100,0.006914,14.808954,8
4,DeepAR,-270.012387,-226.688635,0.032514,0.026609,7.482656,9
5,Toto,-270.801112,-286.002243,1.574893,0.116481,1.430349,17
6,ARIMA,-272.039262,-272.039262,0.013949,0.013277,0.004344,15
7,ETS,-274.007252,-274.007252,0.044029,0.043303,0.004493,4
8,Chronos2,-294.382553,-294.382553,0.891356,0.016221,0.895650,7
9,SimpleFeedForward,-304.340472,-304.340472,0.007795,0.005422,6.504547,16


In [25]:
best_model = leaderboard_monthly.loc[0]
best_model = best_model["model"]
best_model

'WeightedEnsemble'

In [26]:
predictions_monthly = predictor_monthly.predict(
    train_split_monthly, model=best_model
)

In [27]:
selected_item_monthly = "SP500_monthly"

real_data_monthly = test_split_monthly.loc[selected_item_monthly]
predictions_monthly_store = predictions_monthly.loc[selected_item_monthly]

In [28]:
TimeSeriesChart(real_data_monthly,predictions_monthly_store, "Monthly Backtesting - SP500")